# 03 · Data leakage: the 5 types, each demonstrated

**Leakage** = information that would not be available at prediction time sneaks into training or evaluation.
The model looks great offline and fails in production. This is a favourite interview topic.

| # | Type | Example |
|---|---|---|
| 1 | Preprocessing / feature selection fit on all data | Scaling, imputing or selecting features before the split |
| 2 | Target leakage | A feature recorded *after* the outcome (e.g. "antibiotics prescribed" to predict infection) |
| 3 | Group leakage | Same patient in train and test |
| 4 | Temporal leakage | Shuffled split on time-ordered data; training on the future |
| 5 | Duplicates across splits | Augmented / repeated rows land on both sides |

**The fix pattern:** split first → put every fitted step inside a `Pipeline` → choose the splitter
(`GroupKFold`, `TimeSeriesSplit`) that matches how the model will be used.

In [1]:
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold, GroupKFold, TimeSeriesSplit, StratifiedKFold
rng = np.random.default_rng(0)

## Leak 1: feature selection before the split
Pure noise: 100 samples, 10,000 random features, random labels. The honest accuracy is **50%**.

In [2]:
X = rng.normal(size=(100, 10_000))
y = rng.integers(0, 2, 100)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: pick the 20 'best' features using ALL labels, then cross-validate
X_sel = SelectKBest(f_classif, k=20).fit_transform(X, y)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y, cv=cv).mean()

# RIGHT: selection happens inside each training fold
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
right = cross_val_score(pipe, X, y, cv=cv).mean()
print(f"leaky CV accuracy: {wrong:.2f}   honest CV accuracy: {right:.2f}   (truth: 0.50)")

leaky CV accuracy: 0.87   honest CV accuracy: 0.47   (truth: 0.50)


## Leak 2: target leakage
Predict infection. The column `got_antibiotics` is recorded **after** diagnosis, so it encodes the answer.

In [3]:
n = 2000
age = rng.normal(50, 15, n)
crp = rng.normal(10, 5, n)                                  # a real lab marker
logit = -3 + 0.03 * (age - 50) + 0.25 * (crp - 10)
infected = rng.random(n) < 1 / (1 + np.exp(-logit))
got_antibiotics = np.where(infected, rng.random(n) < 0.9, rng.random(n) < 0.05)   # leaky
df = pd.DataFrame({"age": age, "crp": crp, "got_antibiotics": got_antibiotics.astype(int), "infected": infected.astype(int)})

cv = StratifiedKFold(5, shuffle=True, random_state=0)
for cols in [["age", "crp", "got_antibiotics"], ["age", "crp"]]:
    auc = cross_val_score(RandomForestClassifier(random_state=0), df[cols], df.infected, cv=cv, scoring="roc_auc").mean()
    print(f"features={cols}: ROC-AUC {auc:.3f}")

features=['age', 'crp', 'got_antibiotics']: ROC-AUC 0.918


features=['age', 'crp']: ROC-AUC 0.709


The leaky model scores far higher offline, but at prediction time `got_antibiotics` doesn't exist yet.
**Ask for every feature: "when is this value known?"**

## Leak 3: group leakage (same patient in train and test)
Each patient has 10 rows with a personal "signature" and one label. A flexible model can memorise patients.

In [4]:
n_pat, rows = 60, 10
pat_label = rng.integers(0, 2, n_pat)
signature = rng.normal(size=(n_pat, 5))                      # patient-specific, unrelated to label
Xg = np.repeat(signature, rows, axis=0) + rng.normal(0, 0.1, (n_pat * rows, 5))
yg = np.repeat(pat_label, rows)
groups = np.repeat(np.arange(n_pat), rows)

knn = KNeighborsClassifier(n_neighbors=3)
leaky = cross_val_score(knn, Xg, yg, cv=KFold(5, shuffle=True, random_state=0)).mean()
honest = cross_val_score(knn, Xg, yg, cv=GroupKFold(5), groups=groups).mean()
print(f"KFold (patients mixed): {leaky:.2f}   GroupKFold: {honest:.2f}   (truth ≈ 0.50)")

KFold (patients mixed): 1.00   GroupKFold: 0.52   (truth ≈ 0.50)


## Leak 4: temporal leakage
Predict a random-walk value from the time index. Shuffled K-fold lets the model "interpolate" between
neighbouring days, which is impossible when forecasting the future.

In [5]:
t = np.arange(1000).reshape(-1, 1)
series = np.cumsum(rng.normal(size=1000))
reg = KNeighborsRegressor(n_neighbors=5)
shuffled = cross_val_score(reg, t, series, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
forward = cross_val_score(reg, t, series, cv=TimeSeriesSplit(5), scoring="r2").mean()
print(f"R² shuffled KFold: {shuffled:.3f}   R² TimeSeriesSplit: {forward:.3f}")

R² shuffled KFold: 0.978   R² TimeSeriesSplit: -1.389


## Leak 5: duplicates across splits
Copies of the same row on both sides of the split inflate scores (common with augmented images or
repeated records). Deduplicate **before** splitting, or split by the original-item id (GroupKFold).

In [6]:
Xu = rng.normal(size=(300, 5)); yu = rng.integers(0, 2, 300)          # random labels
Xd = np.vstack([Xu, Xu + rng.normal(0, 0.01, Xu.shape)]); yd = np.concatenate([yu, yu])
one_nn = KNeighborsClassifier(n_neighbors=1)
print("with near-duplicates:", cross_val_score(one_nn, Xd, yd, cv=KFold(5, shuffle=True, random_state=0)).mean().round(2))
print("deduplicated        :", cross_val_score(one_nn, Xu, yu, cv=KFold(5, shuffle=True, random_state=0)).mean().round(2))

with near-duplicates: 0.89
deduplicated        : 0.45


## Leakage checklist (memorise)
1. Split **first**; the test set stays locked.
2. Every `fit` (scaler, imputer, encoder, feature selector, PCA, resampler) lives **inside a Pipeline**.
3. For each feature: *when is it known?* Drop anything recorded after the prediction moment.
4. Match the splitter to reality: groups → `GroupKFold`; time → `TimeSeriesSplit`.
5. Deduplicate before splitting.
6. A suspiciously high score is a bug until proven otherwise. Check feature importances for a "too good" feature.

## ✍️ Your turn
1. In Leak 1, change `k` to 200 and 2000. What happens to the leaky score? Why?
2. Find one leakage risk in a past project of yours (heart disease, CSXAI, RAG eval) and write how you'd fix it.

## 🎯 Interview check
- "Your model is 98% accurate offline but 70% in production." Walk through your debugging steps.
- Is fitting `StandardScaler` on the full dataset leakage? How bad is it compared with feature selection?
- How do you split data for a hospital readmission model?

In [7]:
# Your code here